# Attempt at Week2 Challenge
## Transforming Deep Research into an Agentic Flow
### We will attempt to create three top level agents: Preparer, Reporter, and Verifier
Perparer:

    * Generates 3 Questions for the topic
    * Generates 3 Search Strings for the topic


In [51]:
from agents import Agent, Runner, trace, AsyncOpenAI, OpenAIChatCompletionsModel, function_tool, WebSearchTool
from pydantic import BaseModel, Field
from typing import List
import asyncio

### Define the topic to research

In [52]:
topic = "Kobe Bryant"

## Let's define and test a Questioner Agent

In [53]:
# define output types
class Questions(BaseModel):
    questions: List[str]
    """The questions to be asked and tested for"""

questioner_agent_instruction = f"You are given a topic and are tasked with generating a very diffiult question pertaining to that topic."
questioner_agent = Agent(name="questioner", instructions=questioner_agent_instruction, model="gpt-4.1-mini", output_type=Questions)

with trace("Questioner-Test"):
    result = await Runner.run(questioner_agent, topic)
    print(result.final_output)

questions=["Analyze the evolution of Kobe Bryant's playing style over his 20-year NBA career and discuss how his approach to scoring adapted to changes in physical ability and team composition."]


### Define the Search Planner Agent

In [54]:
# define output types
class Search(BaseModel):
    search_string: str = Field("The suggested string to search for")
    reason: str = Field("Why you think this string is valuable to search for")

class SearchPlan(BaseModel):
    searches: List[Search]
    """A list of strings used to search the web for learning purposes"""
    
search_planner_instructions = f"You are a helpful research assistant. Given a topic, come up with a web search string;  Do not actually search the web, just tell me what you would search for on the web"
search_planner_agent = Agent(name="search_planner", instructions=search_planner_instructions, model="gpt-4.1-mini", output_type=SearchPlan)

# test it
with trace("Search-String-Test"):
    result = await Runner.run(search_planner_agent, topic)
    print(result.final_output)

searches=[Search(search_string='Kobe Bryant biography', reason='To learn about his life story and career achievements.'), Search(search_string='Kobe Bryant basketball highlights', reason='To find notable moments and top plays from his basketball career.'), Search(search_string='Kobe Bryant awards and records', reason='To gather information about the awards and records he held during his career.'), Search(search_string='Kobe Bryant legacy and impact on basketball', reason='To understand his influence on the sport and how he is remembered.'), Search(search_string='Kobe Bryant philanthropy and off-court activities', reason='To explore his charitable work and ventures outside basketball.')]


### Create the WebSearch Agent
This agent will search the web on a given topic
Use Ollama for WebSearch to minimize costs (gpt charges more for using the WebSearchTool)

In [55]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
o_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
o_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=o_client)

In [56]:
@function_tool
def custom_websearch_tool():
    """Custom WebSearchTool"""
    return WebSearchTool(search_context_size="low")

In [57]:
web_search_agent_instruction = "You are researcher.  You are provided a topic and are required to use your websearch tool to gather information on the topic"
web_search_agent = Agent(name="search_agent", instructions=web_search_agent_instruction, tools=[custom_websearch_tool], model=o_model)

# test it
with trace("WebSearch-Test"):
    result = await Runner.run(web_search_agent, topic)
    print(result.final_output)

**Kobe Bryant: A Legendary Basketball Player**

Kobe Bean Bryant (August 23, 1978 - January 26, 2020) was an American professional basketball player widely regarded as one of the greatest players in NBA history. He played his entire 20-year career with the Los Angeles Lakers in the National Basketball Association (NBA).

**Early Life and Career**

Bryant was born in Philadelphia, Pennsylvania, to Joe Bryant, a former NBA player, and Pamela Cox. His father's NBA experience had a significant influence on Kobe's early interest in basketball. Bryant began playing organized basketball at age 11 and joined the Junior Olympic Program at 13. By 17, he was drafted by the Charlotte Hornets with the 57th overall pick, but was immediately traded to the Lakers.

**NBA Career**

Bryant made his NBA debut in 1996 as a shooting guard and quickly established himself as one of the league's top players. He won two NBA championships (2000, 2001), one NBA Finals MVP award, and was a 18-time All-Star. Bryan

### Create a Report Generator Agent
This agent will generate a report using the given information

In [58]:
# define output types for the reporter
class Report(BaseModel):
    report: str = Field("The Actual Report")

report_generator_instructions="You are a well known journalist.  You are tasked with taking as input a some data and generating a long, detailed report with atleast 1000 words.  Make this formal, as if millions of people are going to read it."
report_generator_agent = Agent(name="report_generator", instructions=report_generator_instructions, model="gpt-4.1-mini", output_type=Report)

with trace("report-generator-test"):
    result = await Runner.run(report_generator_agent, result.final_output)
    print(result.final_output)

report='Kobe Bryant: A Comprehensive Chronicle of a Basketball Legend\n\nKobe Bean Bryant (August 23, 1978 - January 26, 2020) remains an emblematic figure in the annals of sports history, particularly within the realm of professional basketball. Revered globally, Bryant’s legacy transcends mere athleticism, embodying relentless dedication, profound discipline, and an indefatigable pursuit of excellence — principles collectively celebrated under the banner of the "Mamba Mentality." This detailed report chronicles the extraordinary life, career, achievements, personal journey, and enduring legacy of Kobe Bryant, whose impact continues to resonate across generations.\n\nEarly Life and Formative Years\n\nBorn in Philadelphia, Pennsylvania, Kobe Bryant was imbued with basketball heritage from the outset. His father, Joe Bryant, was himself a professional NBA player, an experience that immensely shaped young Kobe\'s early affinity for the sport. Demonstrating precocious talent, Bryant began

### Create top level Agents
Here, we create the Preparer and Report Agent

In [59]:
# now, the report agent

# define function tool to explicitly orchestrate order of operations
@function_tool
async def reporter_tool(searches: List[str]) -> Report:
    """Report Generator Tool will execute Searches asychronously first, after which it will generate the actual report"""
    # run all searches asynchronously
    coroutines = [Runner.run(web_search_agent, search) for search in searches]
    result = await asyncio.gather(
        *coroutines,
    )

    search_results = '\n\n'.join([r.final_output for r in result])
    # generate the report
    result = await Runner.run(report_generator_agent, "generate a report using the details below: " + search_results)
    return result

# define the report agent
report_agent_instructions = "You are tasked with two things: researching the provided input strings and generating a report with your findings;  Use your tool, reporter_tool, to complete your task."
report_agent = Agent(name="report_generator", instructions=report_agent_instructions, model="gpt-4.1-mini", tools=[reporter_tool])

# some test data
test_search_strings = [
"Kobe Bryant biography",
"Kobe Bryant basketball achievements",
"Kobe Bryant tragic helicopter crash details",
"Kobe Bryant post-retirement activities",
"Kobe Bryant legacy and influence on basketball",
]

with trace("Report-Test"):
    result = await Runner.run(report_agent, " ".join(test_search_strings))
    print(result.final_output)



Here is a comprehensive report on Kobe Bryant covering his biography, basketball achievements, tragic helicopter crash details, post-retirement activities, and his legacy and influence on basketball:

---

**Kobe Bryant: A Comprehensive Tribute to a Basketball Legend and Visionary Entrepreneur**

**Early Life and Formative Years**  
Kobe Bryant was born on August 23, 1978, in Philadelphia, Pennsylvania. The son of former NBA player Joe Bryant, Kobe was introduced to basketball early. His family moved to Italy when he was six months old due to his father's basketball career. Growing up in Rome, Kobe played basketball on the streets and local clubs, developing a passion that would shape his future. Returning to the U.S. at 13, he emerged as a basketball prodigy at Lower Merion High School, earning the title of Mr. Basketball USA in 1996 before entering the NBA directly from high school.

**NBA Career: A Legacy of Excellence**  
Drafted 13th overall by the Charlotte Hornets in 1996 and im

In [60]:
NUM_QUESTIONS = 3
NUM_SEARCH_STRS = 5

preparer_agent_instructions = f"""
You are an assistant preparing your worker to perform some deep research on a topic.
You are tasked with coming up with {NUM_QUESTIONS} very difficult questions related to this topic. 
In addition, you are also tasked with preparing {NUM_SEARCH_STRS} search strings for your worker to use during their research
After you've received all questions and search strings, handoff the data to the reporter agent.  
"""
preparer_agent_tools=[
    questioner_agent.as_tool(tool_name="question_tool", tool_description="this tool is used to generated questions related to a given topic"),
    search_planner_agent.as_tool(tool_name="search_planner_tool", tool_description="this tool is used to generated searching strings for a given topic"),
]
preparer_agent = Agent(name="preparer",
                       instructions=preparer_agent_instructions,
                       model="gpt-4.1-mini",
                       tools=preparer_agent_tools,
                       handoffs=[report_agent],
                       )

# run it !! 
with trace("preparer-agent-test"):
    result = await Runner.run(preparer_agent, topic)
    print(result.final_output)

MaxTurnsExceeded: Max turns (10) exceeded